# Notebook 02 — Dataset and COCO Exploration

## What this notebook does
I use this notebook to download sample video clips for testing, explore the COCO class taxonomy, and set up the data directory structure. I also generate a mock video using synthetic frames so the pipeline can be tested even if real videos have not been downloaded.

## Why this matters
Understanding the COCO class distribution helps me decide which classes to focus on for counting (e.g., vehicles and people for traffic analysis). Knowing the structure of the input videos (resolution, FPS, duration) helps me set realistic benchmarking parameters.

## Input files expected
- `configs/config.yaml`
- Internet connection (to download sample videos)

## Output files created
- `data/raw/sample_traffic.mp4` — sample traffic video
- `data/mock/mock_video.mp4` — synthetic test video
- `data/external/coco_classes.json` — COCO class index → name mapping
- `reports/figures/02_coco_class_categories.png`

## Connection to the research question
The COCO class set determines which real-world objects the system can detect. This notebook maps those classes to the counting task.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

from detection_system.loader import get_coco_classes
from detection_system.utils import load_config, save_figure

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
print("Setup complete")

Setup complete


In [2]:
# ── Step 1: Save COCO class list to external/ ─────────────────────────────────
# I save this as a JSON file so other notebooks and the Streamlit app
# can load it without importing the full detection_system package.

coco_classes = get_coco_classes()
external_dir = PROJECT_ROOT / "data" / "external"
external_dir.mkdir(parents=True, exist_ok=True)

coco_json = {str(i): name for i, name in enumerate(coco_classes)}

with open(external_dir / "coco_classes.json", "w") as f:
    json.dump(coco_json, f, indent=2)

print(f"Saved COCO class list to: data/external/coco_classes.json")
print(f"Total classes: {len(coco_classes)}")

Saved COCO class list to: data/external/coco_classes.json
Total classes: 80


In [3]:
# ── Step 2: Explore COCO class categories ────────────────────────────────────
# COCO classes span multiple real-world domains. I group them here to
# understand what the model can detect and plan the counting focus.

# Manual grouping of COCO 80 classes into semantic categories
category_groups = {
    "People": ["person"],
    "Vehicles": ["bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"],
    "Traffic": ["traffic light", "fire hydrant", "stop sign", "parking meter"],
    "Outdoor": ["bench"],
    "Animals": ["bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe"],
    "Accessories": ["backpack", "umbrella", "handbag", "tie", "suitcase"],
    "Sports": ["frisbee", "skis", "snowboard", "sports ball", "kite",
               "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket"],
    "Kitchen": ["bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl"],
    "Food": ["banana", "apple", "sandwich", "orange", "broccoli", "carrot",
             "hot dog", "pizza", "donut", "cake"],
    "Furniture": ["chair", "couch", "potted plant", "bed", "dining table", "toilet"],
    "Electronics": ["tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
                    "microwave", "oven", "toaster", "sink", "refrigerator"],
    "Household": ["book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"],
}

print(f"Category breakdown ({len(category_groups)} groups):")
for group, classes in category_groups.items():
    print(f"  {group:<14}: {len(classes)} classes — {', '.join(classes[:4])}{'...' if len(classes) > 4 else ''}")

Category breakdown (12 groups):
  People        : 1 classes — person
  Vehicles      : 8 classes — bicycle, car, motorcycle, airplane...
  Traffic       : 4 classes — traffic light, fire hydrant, stop sign, parking meter
  Outdoor       : 1 classes — bench
  Animals       : 10 classes — bird, cat, dog, horse...
  Accessories   : 5 classes — backpack, umbrella, handbag, tie...
  Sports        : 10 classes — frisbee, skis, snowboard, sports ball...
  Kitchen       : 7 classes — bottle, wine glass, cup, fork...
  Food          : 10 classes — banana, apple, sandwich, orange...
  Furniture     : 6 classes — chair, couch, potted plant, bed...
  Electronics   : 11 classes — tv, laptop, mouse, remote...
  Household     : 7 classes — book, clock, vase, scissors...


In [6]:
# ── Step 3: Visualise class category distribution ────────────────────────────
# I use a horizontal bar chart so all category names are readable.

group_sizes = {k: len(v) for k, v in category_groups.items()}
sorted_groups = dict(sorted(group_sizes.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 6))

colors = sns.color_palette("colorblind", n_colors=len(sorted_groups))
bars = ax.barh(
    list(sorted_groups.keys()),
    list(sorted_groups.values()),
    color=colors,
    edgecolor="white",
    linewidth=0.8
)

# Label each bar with its count
for bar, val in zip(bars, sorted_groups.values()):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=10)

ax.set_xlabel("Number of COCO classes", fontsize=12)
ax.set_title("COCO 80-Class Taxonomy — Semantic Category Breakdown", fontsize=13, fontweight="bold")
ax.set_xlim(0, max(sorted_groups.values()) + 2)
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, "02_coco_class_categories")
plt.show()

  Saved: reports\figures\02_coco_class_categories.png
  Saved: reports\figures\02_coco_class_categories.pdf
  Saved: paper\figures\02_coco_class_categories.png
  Saved: paper\figures\02_coco_class_categories.pdf


In [7]:
# ── Step 4: Download a sample traffic video ───────────────────────────────────
# I download a short public-domain traffic video from Pexels (no login required).
# This video is used in all subsequent detection notebooks.
#
# File size: ~15–30 MB. Estimated download time: 10–30 seconds on a fast connection.
# If the download fails, the mock video created in Step 5 can be used instead.

import urllib.request

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_URL = "https://www.pexels.com/download/video/854671/"
VIDEO_PATH = RAW_DIR / "sample_traffic.mp4"

# Alternative public-domain video options (if the above URL fails):
# URL2: "https://sample-videos.com/video321/mp4/720/big_buck_bunny_720p_1mb.mp4"
# URL3: "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4"

if VIDEO_PATH.exists():
    print(f"Video already exists: {VIDEO_PATH}")
    print(f"File size: {VIDEO_PATH.stat().st_size / (1024**2):.1f} MB")
else:
    print(f"Downloading sample video to: {VIDEO_PATH}")
    print("(This may take 10–60 seconds depending on connection speed)")
    
    try:
        # Use a browser-like User-Agent header to avoid 403 errors
        headers = {"User-Agent": "Mozilla/5.0"}
        req = urllib.request.Request(VIDEO_URL, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as resp:
            with open(VIDEO_PATH, "wb") as f:
                f.write(resp.read())
        print(f"Download complete. File size: {VIDEO_PATH.stat().st_size / (1024**2):.1f} MB")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Will use mock video instead (see Step 5).")

(This may take 10–60 seconds depending on connection speed)
Download complete. File size: 34.0 MB


In [8]:
# ── Step 5: Generate a mock video for testing ─────────────────────────────────
# I create a short synthetic video with coloured rectangles moving across
# frames. This is clearly labelled as MOCK DATA and is used ONLY for
# pipeline tests and unit tests — never for real benchmarks or results.
#
# The mock video allows the full pipeline to be tested without downloading
# any real video.

MOCK_DIR = PROJECT_ROOT / "data" / "mock"
MOCK_DIR.mkdir(parents=True, exist_ok=True)
MOCK_VIDEO_PATH = MOCK_DIR / "mock_video.mp4"

# Parameters
MOCK_WIDTH, MOCK_HEIGHT = 640, 480
MOCK_FPS = 25
MOCK_FRAMES = 100  # 4 seconds at 25 FPS

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(MOCK_VIDEO_PATH), fourcc, MOCK_FPS, (MOCK_WIDTH, MOCK_HEIGHT))

np.random.seed(42)  # Fixed seed for reproducibility

for frame_idx in range(MOCK_FRAMES):
    # Grey background
    frame = np.full((MOCK_HEIGHT, MOCK_WIDTH, 3), 128, dtype=np.uint8)
    
    # Moving coloured rectangle (simulates an object)
    x = int(50 + (frame_idx / MOCK_FRAMES) * (MOCK_WIDTH - 100))
    cv2.rectangle(frame, (x, 150), (x + 80, 250), (86, 180, 233), -1)
    
    # Static rectangle
    cv2.rectangle(frame, (400, 300), (550, 400), (230, 159, 0), -1)
    
    # Frame number label
    cv2.putText(frame, f"[MOCK DATA] Frame {frame_idx}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    writer.write(frame)

writer.release()
print(f"Mock video created: {MOCK_VIDEO_PATH}")
print(f"  Frames   : {MOCK_FRAMES}")
print(f"  FPS      : {MOCK_FPS}")
print(f"  Duration : {MOCK_FRAMES / MOCK_FPS:.1f} seconds")
print(f"  Size     : {MOCK_VIDEO_PATH.stat().st_size / 1024:.1f} KB")
print("\nNOTE: This is MOCK DATA — used only for pipeline tests, not real analysis.")

Mock video created: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\mock\mock_video.mp4
  Frames   : 100
  FPS      : 25
  Duration : 4.0 seconds
  Size     : 129.9 KB

NOTE: This is MOCK DATA — used only for pipeline tests, not real analysis.


In [9]:
# ── Step 6: Inspect available videos ─────────────────────────────────────────
# I list all video files found in data/raw/ and data/mock/ so I know
# what's available for subsequent notebooks.

def inspect_video(path):
    """Print key metadata for a video file."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"  ERROR: Cannot open {path.name}")
        return
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_s = n_frames / fps if fps > 0 else 0
    cap.release()
    
    print(f"  {path.name}")
    print(f"    Resolution : {width}×{height} px")
    print(f"    FPS        : {fps:.1f}")
    print(f"    Frames     : {n_frames}")
    print(f"    Duration   : {duration_s:.1f} s")
    print(f"    File size  : {path.stat().st_size / (1024**2):.1f} MB")

print("=== Videos in data/raw/ ===")
raw_videos = list((PROJECT_ROOT / "data" / "raw").glob("*.mp4"))
if raw_videos:
    for v in raw_videos:
        inspect_video(v)
else:
    print("  No real videos found — use mock video for testing.")

print("\n=== Videos in data/mock/ ===")
for v in (PROJECT_ROOT / "data" / "mock").glob("*.mp4"):
    inspect_video(v)

=== Videos in data/raw/ ===
  sample_traffic.mp4
    Resolution : 1920×1080 px
    FPS        : 25.0
    Frames     : 1501
    Duration   : 60.0 s
    File size  : 34.0 MB

=== Videos in data/mock/ ===
  mock_video.mp4
    Resolution : 640×480 px
    FPS        : 25.0
    Frames     : 100
    Duration   : 4.0 s
    File size  : 0.1 MB
